In [2]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, save, output_file
from bokeh.layouts import gridplot
from bokeh.models import HoverTool, Select, CustomJS, ColumnDataSource
from bokeh.palettes import Category20
from bokeh.transform import cumsum, factor_cmap
import math
import random  # AÑADIR ESTA LÍNEA

# Crear datos demográficos simulados realistas
np.random.seed(123)

# Datos de población por regiones
regiones = ['Norte América', 'Europa', 'Asia-Pacífico', 'América Latina', 'África', 'Medio Oriente']
años = list(range(2000, 2025))

datos_poblacion = []
for region in regiones:
    pop_inicial = random.randint(500, 4500)  # Millones
    crecimiento_anual = random.uniform(0.005, 0.035)  # 0.5% a 3.5%
    
    for año in años:
        años_transcurridos = año - 2000
        poblacion = pop_inicial * (1 + crecimiento_anual) ** años_transcurridos
        
        # Añadir variabilidad realista
        variabilidad = random.uniform(0.95, 1.05)
        poblacion *= variabilidad
        
        datos_poblacion.append({
            'region': region,
            'año': año,
            'poblacion_millones': round(poblacion, 1),
            'pib_per_capita': random.randint(5000, 65000),
            'esperanza_vida': random.uniform(65, 85),
            'educacion_superior_pct': random.uniform(15, 75)
        })

df_demografico = pd.DataFrame(datos_poblacion)

def crear_analisis_demografico():
    # GRÁFICO 1: Evolución temporal de población
    p1 = figure(
        title="🌍 Evolución de la Población Mundial por Región (2000-2024)",
        x_axis_label="Año",
        y_axis_label="Población (Millones)",
        width=1000,
        height=400,
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    
    colores = Category20[len(regiones)]
    
    for i, region in enumerate(regiones):
        data = df_demografico[df_demografico['region'] == region]
        
        p1.line(data['año'], data['poblacion_millones'],
               line_width=3,
               color=colores[i],
               legend_label=region,
               alpha=0.8)
        
        # CAMBIO: Usar scatter en lugar de circle para evitar warnings
        p1.scatter(data['año'], data['poblacion_millones'],
                  size=6,
                  color=colores[i],
                  alpha=0.7)
    
    # Estilización profesional
    p1.title.text_font_size = "16pt"
    p1.title.text_color = "#34495e"
    p1.legend.location = "top_left"
    p1.legend.click_policy = "hide"
    p1.background_fill_color = "#f8f9fa"
    
    # GRÁFICO 2: Relación PIB vs Esperanza de Vida
    p2 = figure(
        title="💰 PIB per Cápita vs Esperanza de Vida (2024)",
        x_axis_label="PIB per Cápita (USD)",
        y_axis_label="Esperanza de Vida (años)",
        width=600,
        height=500,
        tools="pan,wheel_zoom,box_select,reset,save"
    )
    
    # Datos más recientes (2024)
    data_2024 = df_demografico[df_demografico['año'] == 2024]
    
    # Crear ColumnDataSource para mejor hover
    source_2024 = ColumnDataSource(data_2024)
    
    # Crear scatter plot con tamaño basado en población
    for i, region in enumerate(regiones):
        region_data = data_2024[data_2024['region'] == region]
        
        p2.scatter(region_data['pib_per_capita'], 
                  region_data['esperanza_vida'],
                  size=region_data['poblacion_millones']/50,  # Escalar tamaño
                  color=colores[i],
                  alpha=0.7,
                  legend_label=region)
    
    # Hover tool personalizado mejorado
    hover = HoverTool(tooltips=[
        ("Región", "@region"),
        ("PIB per Cápita", "$@pib_per_capita{0,0}"),
        ("Esperanza de Vida", "@esperanza_vida{0.0} años"),
        ("Población", "@poblacion_millones{0.0}M"),
        ("Educación Superior", "@educacion_superior_pct{0.0}%")
    ])
    
    # Añadir scatter con source para hover
    p2.scatter('pib_per_capita', 'esperanza_vida',
              size='poblacion_millones',
              source=source_2024,
              alpha=0)  # Invisible, solo para hover
    
    p2.add_tools(hover)
    
    p2.title.text_font_size = "14pt"
    p2.title.text_color = "#34495e"
    p2.legend.location = "bottom_right"
    
    # GRÁFICO 3: Distribución de Educación Superior
    p3 = figure(
        title="🎓 Distribución de Educación Superior por Región",
        x_range=regiones,
        width=800,
        height=400,
        tools="save"
    )
    
    educacion_2024 = data_2024['educacion_superior_pct'].values
    
    p3.vbar(x=regiones, top=educacion_2024, width=0.8,
           color=colores[:len(regiones)], alpha=0.8)
    
    p3.xgrid.grid_line_color = None
    p3.y_range.start = 0
    p3.xaxis.major_label_orientation = math.pi/4
    p3.yaxis.axis_label = "Porcentaje de Población (%)"
    
    # Estilización adicional
    p3.title.text_font_size = "14pt"
    p3.title.text_color = "#34495e"
    
    # Layout en grid
    grid = gridplot([[p1], [p2, p3]], sizing_mode="scale_width")
    
    return grid

# Crear y guardar análisis demográfico
analisis_demo = crear_analisis_demografico()
output_file("html/analisis_demografico.html")
save(analisis_demo)

print("✅ Análisis Demográfico Profesional creado: html/analisis_demografico.html")
print("🌍 Gráficos demográficos listos para portafolio!")

✅ Análisis Demográfico Profesional creado: html/analisis_demografico.html
🌍 Gráficos demográficos listos para portafolio!
